# Forward to the Past — Création du catalogue

Ce notebook construit un catalogue de films à partir des données IMDb, TMDB et Entrées France. Il couvre l’acquisition des sources, leur préparation, les jointures et enrichissements, la sélection métier et l’export du catalogue au format CSV.

Les archives brutes IMDb et TMDB restent hors Git. Les dossiers du projet doivent exister et les fichiers TMDB et Entrées France doivent être déposés aux emplacements indiqués en section 0.2.

Exécuter les cellules dans l’ordre depuis un kernel vierge : certains traitements modifient les DataFrames en place.

## 0 — Imports et configuration

Les paramètres techniques et les critères métier sont regroupés ici pour faciliter les modifications.

### 0.1 — Imports

In [1]:
import csv
from pathlib import Path
from time import perf_counter

import pandas as pd
import requests
from IPython.display import display

### 0.2 — Configuration technique

La racine du dépôt correspond au parent du répertoire de travail (exécution locale depuis `notebooks/`). Pour une autre organisation, notamment dans Colab, adapter `RACINE_PROJET` et les chemins si nécessaire.

`FORCER_TELECHARGEMENT` renouvelle les fichiers IMDb déjà présents. `TAILLE_BLOC` et `FREQUENCE_PROGRESSION` règlent la lecture par blocs et l’affichage de la progression.

In [ ]:
FORCER_TELECHARGEMENT = False

RACINE_PROJET = Path.cwd().parent
DOSSIER_RAW = RACINE_PROJET / 'data/raw'
DOSSIER_PROCESSED = RACINE_PROJET / 'data/processed'
DOSSIER_SUPPLEMENTS = RACINE_PROJET / 'data/supplements'

FICHIER_ENTREES = 'Entrees_France.csv'
FICHIER_TMDB = 'TMDB_movie_dataset_v11.csv'
FICHIER_CATALOGUE = 'catalogue.csv'
FICHIER_IMDB_BASICS = 'title.basics.tsv.gz'
FICHIER_IMDB_RATINGS = 'title.ratings.tsv.gz'
FICHIER_IMDB_AKAS = 'title.akas.tsv.gz'

TAILLE_BLOC = 200_000
FREQUENCE_PROGRESSION = 10

SOURCE_IMDB = 'https://datasets.imdbws.com/'
NOMS_IMDB = [
    'name.basics.tsv.gz',
    FICHIER_IMDB_AKAS,
    FICHIER_IMDB_BASICS,
    'title.principals.tsv.gz',
    FICHIER_IMDB_RATINGS,
]

Les chemins, options de lecture et colonnes utiles sont définis une seule fois. Seuls les trois jeux de données IMDb exploités sont chargés en mémoire ; les cinq fichiers sont néanmoins acquis. Les colonnes obligatoires d’Entrées France définissent le schéma attendu par la préparation.

In [ ]:
CHEMIN_IMDB_BASICS = DOSSIER_RAW / FICHIER_IMDB_BASICS
CHEMIN_IMDB_RATINGS = DOSSIER_RAW / FICHIER_IMDB_RATINGS
CHEMIN_IMDB_AKAS = DOSSIER_RAW / FICHIER_IMDB_AKAS
CHEMIN_ENTREES_FRANCE = DOSSIER_SUPPLEMENTS / FICHIER_ENTREES
CHEMIN_TMDB = DOSSIER_RAW / FICHIER_TMDB
CHEMIN_CATALOGUE = DOSSIER_PROCESSED / FICHIER_CATALOGUE

OPTIONS_IMDB = {
    'sep': '\t',
    'compression': 'gzip',
    'quoting': csv.QUOTE_NONE,
    'dtype': 'string',
    'na_values': [r'\N'],
}

OPTIONS_ENTREES_FRANCE = {
    'sep': ';',
    'encoding': 'utf-8-sig',
    'dtype': 'string',
}

OPTIONS_TMDB = {
    'dtype': 'string',
    'na_values': [''],
}

COLONNES_BASICS = [
    'tconst', 'titleType', 'primaryTitle', 'originalTitle',
    'isAdult', 'startYear', 'runtimeMinutes', 'genres',
]
COLONNES_RATINGS = ['tconst', 'averageRating', 'numVotes']
COLONNES_TMDB = [
    'id', 'imdb_id', 'title', 'original_title', 'original_language',
    'production_countries', 'release_date', 'vote_average', 'vote_count',
    'popularity', 'budget', 'revenue', 'overview', 'keywords', 'poster_path',
]
COLONNES_AKAS = ['titleId', 'ordering', 'title', 'region', 'types']

COLONNES_REQUISES_ENTREES_FRANCE = {
    'imdb_id',
    'tmdb_id',
    'jp_id',
    'jp_id_film',
    'est_reprise',
    'saga',
    'genre_source',
    'entrees_france_cumulees',
    'statut_cumul',
    'statut_jointure',
    'controle_oeuvre',
    'controle_concordance',
    'url_source_cumul',
    'url_source_saga',
    'date_collecte',
}

### 0.3 — Configuration métier

Les types de titres, régions et statuts autorisés définissent la présélection et la préparation des sources. `FILTRES_METIER` règle la sélection finale ; `COLONNES_EXPORT` fixe le schéma et l’ordre des colonnes du CSV.

In [4]:
TYPES_TITRES_AUTORISES = ["movie"]
REGIONS_AUTORISEES = ["FR"]
STATUTS_OEUVRE_EXCLUS = ['edition_ou_programme_a_verifier', 'imdb_non_confirme']

STATUTS_JOINTURE_ACCEPTES = [
    'accepte_score',
    'accepte_consolide',
    'accepte_wikidata',
    'accepte_consolide_reprise',
    'accepte_corrige_homonyme',
]

STATUTS_CUMUL_ACCEPTES = {'total_publie_jp', 'exploitations_observees', 'sur_ligne_du_film'}

In [5]:
FILTRES_METIER = {
    # Année de sortie
    'annee_min': 1960,
    'annee_max': 2016,

    # Durée en minutes
    'duree_min': 60,
    'duree_max': 255,

    # Entrées France
    'entrees_min': 150_000,
    'entrees_max': None,
    'garder_entrees_inconnues': False,

    # Genres Entrées France
    'genres_inclus': [
        'Animation',
        'Aventure - Action',
        'Aventure - action',
        'Catastrophe',
        'Comicbook',
        'Comédie',
        'Comédie dramatique',
        'Court-métrage',
        'Documentaire',
        'Drame',
        'Fantasy',
        'Film familial',
        'Film religieux',
        'Guerre',
        'Horreur',
        'Musical',
        'Péplum',
        'Romance',
        'Science Fiction',
        'Thriller',
        'Western',
    ],
    'genres_exclus': ['X - Erotique'],

    # Langues et origines
    'langues_exclues': [],
    'origines': [],

    # Nombre de votes IMDb
    'votes_min': None,
    'votes_min_fr': None,

    # Critères complémentaires
    'non_adultes': True,
    'exiger_tmdb': False,
    'exiger_visuels': False,
}

In [6]:
COLONNES_EXPORT = [
    # Identifiants et titres
    'tconst',
    'tmdb_id',
    'titre_affiche',
    'originalTitle',

    # Caractéristiques et production
    'startYear',
    'runtimeMinutes',
    'genre_source',
    'original_language',
    'origine_production',
    'production_countries',
    'film_francais',

    # Entrées France, sagas et reprises
    'entrees_france',
    'type_cumul_entrees',
    'saga',

    # Notes et votes IMDb
    'averageRating',
    'numVotes',

    # Résumé et affiche TMDB
    'overview',
    'poster_path',
]

### 0.4 — Fonction utilitaire transversale

`champ_vide()` reconnaît les valeurs manquantes et leurs représentations textuelles. Zéro reste une valeur valide.

In [7]:
def champ_vide(serie):
    texte = (
        serie.astype('string')
        .str.strip()
        .str.casefold()
    )
    return serie.isna() | texte.isin(['', 'null', 'none', 'nan', r'\n', '[]', '{}'])

## 1 — Acquisition et chargement des sources

Les sources sont récupérées puis présélectionnées en mémoire. Aucune jointure ni filtre final n’est effectué à cette étape.

### 1.1 — Vérification des sources locales

Vérifier la présence des fichiers TMDB et Entrées France avant
de procéder au téléchargement des archives IMDb.

In [ ]:
for fichier in [CHEMIN_ENTREES_FRANCE, CHEMIN_TMDB]:
    if not fichier.is_file():
        raise FileNotFoundError(f"Fichier source manquant : {fichier}")

    print(f"[OK] {fichier.name} : disponible")

### 1.2 — Acquisition des fichiers IMDb

Télécharger les fichiers absents ou renouveler les fichiers existants si demandé. Un fichier `.part` est utilisé pendant le téléchargement pour éviter de réutiliser un fichier incomplet.

In [8]:
print("[Acquisition] Fichiers IMDb", flush=True)

for nom in NOMS_IMDB:
    lien = SOURCE_IMDB + nom
    fichier = DOSSIER_RAW / nom
    fichier_temporaire = DOSSIER_RAW / (nom + ".part")

    if fichier.exists() and not FORCER_TELECHARGEMENT:
        print(f"[IGNORÉ] {nom} : déjà présent")
        continue

    print(f"[TÉLÉCHARGEMENT] {nom}", flush=True)
    debut = perf_counter()

    with requests.get(lien, stream=True, timeout=15) as reponse:
        reponse.raise_for_status()

        with fichier_temporaire.open("wb") as f:
            for morceau in reponse.iter_content(chunk_size=8192):
                if morceau:
                    f.write(morceau)

    fichier_temporaire.replace(fichier)

    print(f"[OK] {nom} | {perf_counter() - debut:.1f} s", flush=True)

[Acquisition] Fichiers IMDb
[IGNORÉ] name.basics.tsv.gz : déjà présent
[IGNORÉ] title.akas.tsv.gz : déjà présent
[IGNORÉ] title.basics.tsv.gz : déjà présent
[IGNORÉ] title.principals.tsv.gz : déjà présent
[IGNORÉ] title.ratings.tsv.gz : déjà présent


### 1.3 — Vérification des fichiers IMDb

Vérifier la présence des cinq fichiers IMDb après leur acquisition.

In [ ]:
for nom in NOMS_IMDB:
    fichier = DOSSIER_RAW / nom

    if not fichier.is_file():
        raise FileNotFoundError(f"Fichier manquant : {fichier}")

print("Toutes les sources attendues sont disponibles.")

Toutes les sources attendues sont disponibles.


### 1.4 — Chargement en mémoire

Les cinq sources exploitées sont chargées avec une présélection adaptée. Les identifiants des films IMDb servent à limiter les volumes des sources associées.

`lire_par_blocs()` filtre les fichiers volumineux au fil de la lecture et affiche la progression.

In [10]:
def lire_par_blocs(adresse, colonnes, retenir, *, taille_bloc, frequence_progression, **options):
    morceaux = []
    total = 0
    total_retenu = 0
    debut = perf_counter()

    with pd.read_csv(
        adresse,
        usecols=colonnes,
        chunksize=taille_bloc,
        **options,
    ) as lecteur:
        for numero, bloc in enumerate(lecteur, start=1):
            total += len(bloc)
            extrait = bloc.loc[retenir(bloc)].copy()

            if not extrait.empty:
                morceaux.append(extrait)
                total_retenu += len(extrait)

            if numero % frequence_progression == 0:
                print(
                    f"  Bloc {numero} | {total:,} lignes lues | "
                    f"{total_retenu:,} retenues | {perf_counter() - debut:.1f} s",
                    flush=True,
                )

    print('  Assemblage des blocs retenus…', flush=True)
    if morceaux:
        table = pd.concat(morceaux, ignore_index=True)
    else:
        table = pd.DataFrame(columns=colonnes)

    print(
        f"  [OK] {total:,} lignes lues | {len(table):,} retenues | "
        f"{perf_counter() - debut:.1f} s",
        flush=True,
    )
    return table, total

#### 1.4.1 — IMDb Basics

Conserver les types de titres autorisés et les huit colonnes utiles. Le nombre total de lignes lues est conservé avant présélection.

In [ ]:
print('[1.4.1 — 1/5] IMDb Basics — films', flush=True)
films, total_films = lire_par_blocs(
    CHEMIN_IMDB_BASICS, COLONNES_BASICS,
    lambda bloc: bloc.titleType.isin(TYPES_TITRES_AUTORISES),
    taille_bloc=TAILLE_BLOC,
    frequence_progression=FREQUENCE_PROGRESSION,
    **OPTIONS_IMDB,
)

[1.3.1 — 1/5] IMDb Basics — films
  Bloc 10 | 2,000,000 lignes lues | 267,030 retenues | 2.9 s
  Bloc 20 | 4,000,000 lignes lues | 344,711 retenues | 5.5 s
  Bloc 30 | 6,000,000 lignes lues | 431,378 retenues | 8.3 s
  Bloc 40 | 8,000,000 lignes lues | 522,268 retenues | 11.0 s
  Bloc 50 | 10,000,000 lignes lues | 618,793 retenues | 13.8 s
  Bloc 60 | 12,000,000 lignes lues | 721,479 retenues | 16.7 s
  Assemblage des blocs retenus…
  [OK] 13,003,096 lignes lues | 761,872 retenues | 18.1 s


#### 1.4.2 — IMDb Ratings

Conserver les notes et votes associés aux identifiants des films retenus.

In [ ]:
ids_films = set(films.tconst)

print('[1.4.2 — 2/5] IMDb Ratings — notes et votes', flush=True)
notes, total_notes = lire_par_blocs(
    CHEMIN_IMDB_RATINGS, COLONNES_RATINGS,
    lambda bloc: bloc.tconst.isin(ids_films),
    taille_bloc=TAILLE_BLOC,
    frequence_progression=FREQUENCE_PROGRESSION,
    **OPTIONS_IMDB,
)

[1.3.2 — 2/5] IMDb Ratings — notes et votes
  Assemblage des blocs retenus…
  [OK] 1,736,437 lignes lues | 352,335 retenues | 48.9 s


#### 1.4.3 — Entrées France

Charger intégralement le CSV des entrées françaises, sans filtre à ce stade.

In [ ]:
print('[1.4.3 — 3/5] Entrées France', flush=True)
debut = perf_counter()
entrees = pd.read_csv(CHEMIN_ENTREES_FRANCE, **OPTIONS_ENTREES_FRANCE)
total_entrees_france = len(entrees)
print(f"  [OK] {total_entrees_france:,} lignes chargées | {perf_counter() - debut:.1f} s", flush=True)

[1.3.3 — 3/5] Entrées France
  [OK] 17,597 lignes chargées | 0.2 s


#### 1.4.4 — TMDB

Conserver une fiche si son identifiant IMDb correspond à un film retenu **ou** si son identifiant TMDB figure dans Entrées France. Les jointures contrôlent ensuite la validité de ces correspondances.

In [ ]:
print('[1.4.4 — 4/5] TMDB', flush=True)
ids_entrees = set(entrees['tmdb_id'].dropna())
tmdb, total_tmdb = lire_par_blocs(
    CHEMIN_TMDB, COLONNES_TMDB,
    lambda bloc: (
        bloc.imdb_id.str.strip().isin(ids_films)
        | bloc.id.isin(ids_entrees)
    ),
    taille_bloc=TAILLE_BLOC,
    frequence_progression=FREQUENCE_PROGRESSION,
    **OPTIONS_TMDB,
)

[1.3.4 — 4/5] TMDB
  Assemblage des blocs retenus…
  [OK] 1,529,947 lignes lues | 366,494 retenues | 50.3 s


#### 1.4.5 — IMDb Akas

Conserver les titres des régions autorisées, puis limiter les résultats aux identifiants des films retenus.

In [ ]:
print('[1.4.5 — 5/5] IMDb Akas — titres', flush=True)

titres, total_titres = lire_par_blocs(
    CHEMIN_IMDB_AKAS, COLONNES_AKAS,
    lambda bloc: bloc.region.isin(REGIONS_AUTORISEES),
    taille_bloc=TAILLE_BLOC,
    frequence_progression=FREQUENCE_PROGRESSION,
    **OPTIONS_IMDB,
)

print('Filtrage des titres par identifiants IMDb…', flush=True)
titres = titres.loc[titres.titleId.isin(ids_films)].copy()

print(f'Titres retenus : {len(titres):,}')

[1.3.5 — 5/5] IMDb Akas — titres
  Bloc 10 | 2,000,000 lignes lues | 63,837 retenues | 1.6 s
  Bloc 20 | 4,000,000 lignes lues | 175,047 retenues | 3.1 s
  Bloc 30 | 6,000,000 lignes lues | 364,223 retenues | 4.6 s
  Bloc 40 | 8,000,000 lignes lues | 567,010 retenues | 6.1 s
  Bloc 50 | 10,000,000 lignes lues | 773,728 retenues | 7.5 s
  Bloc 60 | 12,000,000 lignes lues | 977,669 retenues | 9.0 s
  Bloc 70 | 14,000,000 lignes lues | 1,181,285 retenues | 10.5 s
  Bloc 80 | 16,000,000 lignes lues | 1,381,341 retenues | 11.9 s
  Bloc 90 | 18,000,000 lignes lues | 1,582,852 retenues | 13.3 s
  Bloc 100 | 20,000,000 lignes lues | 1,779,892 retenues | 14.8 s
  Bloc 110 | 22,000,000 lignes lues | 1,973,437 retenues | 16.3 s
  Bloc 120 | 24,000,000 lignes lues | 2,167,404 retenues | 17.8 s
  Bloc 130 | 26,000,000 lignes lues | 2,365,457 retenues | 19.3 s
  Bloc 140 | 28,000,000 lignes lues | 2,559,380 retenues | 20.7 s
  Bloc 150 | 30,000,000 lignes lues | 2,751,023 retenues | 22.2 s
  Bloc 16

#### 1.4.6 — Inventaire des sources

Récapituler les volumes lus et retenus.

In [16]:
inventaire = pd.DataFrame([
    ['IMDb Basics', total_films, len(films), len(films.columns)],
    ['IMDb Ratings', total_notes, len(notes), len(notes.columns)],
    ['Entrées France', total_entrees_france, len(entrees), len(entrees.columns)],
    ['TMDB', total_tmdb, len(tmdb), len(tmdb.columns)],
    ['IMDb Akas', total_titres, len(titres), len(titres.columns)],
], columns=[
    'Source',
    'Lignes lues',
    'Lignes retenues',
    'Nombre de colonnes',
])

display(inventaire)

print("[OK] Acquisition et chargement terminés. Les cinq sources sont en mémoire.")

,Source,Lignes lues,Lignes retenues,Nombre de colonnes
0,IMDb Basics,13003096,761872,8
1,IMDb Ratings,1736437,352335,3
2,Entrées France,17597,17597,31
3,TMDB,1529947,366494,15
4,IMDb Akas,60666746,106162,5


[OK] Acquisition et chargement terminés. Les cinq sources sont en mémoire.


## 2 — Périmètre de l’exploration

Le pipeline s’appuie sur les schémas et les règles de préparation définis lors de l’exploration des sources. Les diagnostics exploratoires sont distincts des traitements de construction du catalogue.

## 3 — Préparation individuelle des sources

Chaque source est préparée avant les jointures : conversion des types, sélection d’une ligne par identifiant lorsque nécessaire et consolidation des exploitations d’Entrées France. Les critères de sélection métier s’appliquent au catalogue enrichi en section 6.

`verifier_ids()` contrôle la présence et l’unicité des identifiants IMDb dans les tables préparées.

In [17]:
def verifier_ids(frame, nom):
    if frame['tconst'].isna().any() or frame['tconst'].duplicated().any():
        raise ValueError(nom + ' : IMDb absent ou dupliqué')

### 3.1 — IMDb Basics

Vérifier les identifiants IMDb et convertir les indicateurs, années et durées en numérique.

In [18]:
print("[3.1 — IMDb Basics] Préparation…", flush=True)
debut = perf_counter()

verifier_ids(films, "title.basics")
for colonne in ['isAdult', 'startYear', 'runtimeMinutes']:
    films[colonne] = pd.to_numeric(films[colonne], errors='coerce')

print(f"[OK] {len(films):,} lignes préparées | {perf_counter() - debut:.1f} s", flush=True)

[3.1 — IMDb Basics] Préparation…
[OK] 761,872 lignes préparées | 0.9 s


### 3.2 — IMDb Ratings

Vérifier les identifiants puis convertir les notes et le nombre de votes en numérique.

In [19]:
print("[3.2 — IMDb Ratings] Préparation…", flush=True)
debut = perf_counter()

verifier_ids(notes, 'title.ratings')
for colonne in ['averageRating', 'numVotes']:
    notes[colonne] = pd.to_numeric(notes[colonne], errors='coerce')

print(f"[OK] {len(notes):,} lignes préparées | {perf_counter() - debut:.1f} s", flush=True)

[3.2 — IMDb Ratings] Préparation…
[OK] 352,335 lignes préparées | 0.3 s


### 3.3 — TMDB

Conserver une fiche par IMDb, en privilégiant le nombre de votes le plus élevé. Le DataFrame `tmdb` reste disponible pour les compléments ultérieurs.

In [20]:
print("[3.3 — TMDB] Préparation…", flush=True)
debut = perf_counter()

tmdb_prepare = tmdb.loc[tmdb['imdb_id'].isin(ids_films)].copy()
tmdb_prepare['vote_count'] = pd.to_numeric(tmdb_prepare['vote_count'], errors='coerce')
tmdb_prepare = tmdb_prepare.sort_values('vote_count', ascending=False, kind='mergesort')
tmdb_prepare = tmdb_prepare.drop_duplicates('imdb_id')

tmdb_prepare = tmdb_prepare.rename(columns={
    'id': 'tmdb_id',
    'title': 'titre_tmdb',
    'vote_count': 'votes_tmdb',
    'vote_average': 'note_tmdb',
})

print(f"[OK] {len(tmdb_prepare):,} lignes préparées | {perf_counter() - debut:.1f} s", flush=True)

[3.3 — TMDB] Préparation…
[OK] 363,832 lignes préparées | 5.9 s


### 3.4 — IMDb Akas

Sélectionner un titre français par film : priorité à `imdbDisplay`, puis à la valeur `ordering` la plus faible.

In [21]:
print("[3.4 — IMDb Akas] Préparation…", flush=True)
debut = perf_counter()

titres['types'] = titres['types'].fillna("")
titres['_priorite'] = titres['types'].eq("imdbDisplay")
titres['ordering'] = pd.to_numeric(titres['ordering'], errors='coerce')
titres = titres.sort_values(
    ['_priorite', 'ordering'],
    ascending=[False, True],
    na_position='last',
)
titres = titres.drop_duplicates('titleId')
titres = titres[['titleId', 'title']].rename(columns={'title': 'titre_fr'})

print(f"[OK] {len(titres):,} lignes préparées | {perf_counter() - debut:.1f} s", flush=True)

[3.4 — IMDb Akas] Préparation…
[OK] 95,500 lignes préparées | 0.1 s


### 3.5 — Entrées France

Les correspondances sont contrôlées et les exploitations consolidées à la granularité d’un film IMDb. Le DataFrame brut `entrees` est conservé pour les compléments TMDB de la section 4.5.

#### 3.5.1 — Nettoyage

Vérifier les colonnes nécessaires, retirer les espaces superflus et convertir les entrées cumulées et l’indicateur de reprise.

In [22]:
print("[3.5.1] Nettoyage d’Entrées France…", flush=True)

entrees_prepare = entrees.copy()

if not COLONNES_REQUISES_ENTREES_FRANCE.issubset(entrees_prepare.columns):
    raise ValueError(
        'Entrees_France.csv : colonnes manquantes '
        + str(sorted(COLONNES_REQUISES_ENTREES_FRANCE - set(entrees_prepare.columns))),
    )

for colonne in entrees_prepare:
    entrees_prepare[colonne] = entrees_prepare[colonne].str.strip()

entrees_prepare['motif_ecart'] = ""
entrees_prepare['entrees_france_cumulees'] = pd.to_numeric(
    entrees_prepare['entrees_france_cumulees'], errors='coerce',
)
entrees_prepare['est_reprise'] = (
    entrees_prepare['est_reprise']
    .map({'True': True, 'False': False})
    .astype('boolean')
)

print(f"[OK] {len(entrees_prepare):,} lignes nettoyées", flush=True)

[3.5.1] Nettoyage d’Entrées France…
[OK] 17,597 lignes nettoyées


#### 3.5.2 — Correspondances acceptées

Signaler les statuts non acceptés, les identifiants invalides ou obligatoires manquants et les œuvres incertaines. Les motifs successifs peuvent remplacer le motif précédent.

In [23]:
print("[3.5.2] Contrôle des correspondances…", flush=True)

masque_refus = ~entrees_prepare['statut_jointure'].isin(STATUTS_JOINTURE_ACCEPTES)
entrees_prepare.loc[masque_refus, 'motif_ecart'] = 'Correspondance non acceptée dans la source'

masque = ~entrees_prepare['imdb_id'].str.fullmatch(r'tt\d+', na=False)
entrees_prepare.loc[masque, 'motif_ecart'] = 'Identifiant IMDb absent ou invalide'

masque_oeuvre_incertaine = entrees_prepare['controle_oeuvre'].isin(STATUTS_OEUVRE_EXCLUS)
entrees_prepare.loc[masque_oeuvre_incertaine, 'motif_ecart'] = 'Œuvre ou édition à vérifier'

masque = (
    entrees_prepare['tmdb_id'].notna()
    & ~entrees_prepare['tmdb_id'].str.fullmatch(r'\d+')
)
entrees_prepare.loc[masque, 'motif_ecart'] = 'Identifiant TMDB invalide'

masque = (entrees_prepare['jp_id'].isna()) | (entrees_prepare['jp_id'].eq(""))
entrees_prepare.loc[masque, 'motif_ecart'] = 'Exploitation JP sans identifiant'

print(f"[OK] {entrees_prepare.motif_ecart.ne('').sum():,} lignes signalées", flush=True)

[3.5.2] Contrôle des correspondances…
[OK] 128 lignes signalées


#### 3.5.3 — Doublons et conflits d’exploitation

Supprimer les lignes identiques, puis signaler les identifiants JP répétés dans des lignes différentes. Les correspondances écartées sont conservées pour le bilan.

In [24]:
print("[3.5.3] Contrôle des doublons JP…", flush=True)

entrees_prepare = entrees_prepare.drop_duplicates()
masque_conflit_jp = (
    entrees_prepare['jp_id'].notna()
    & entrees_prepare['jp_id'].ne('')
    & entrees_prepare['jp_id'].duplicated(keep=False)
)
entrees_prepare.loc[masque_conflit_jp, 'motif_ecart'] = 'Identifiant exploitation JP répété avec des données différentes'

print(f"[OK] {len(entrees_prepare):,} lignes après dédoublonnage", flush=True)

[3.5.3] Contrôle des doublons JP…
[OK] 17,597 lignes après dédoublonnage


#### 3.5.4 — Consolidation par film IMDb

Séparer les lignes valides des écarts, puis regrouper les exploitations par IMDb. Les groupes présentant des identifiants TMDB ou JP Film contradictoires sont écartés. Un cumul d’entrées cohérent est retenu sans additionner les montants répétés. Les sagas, genres, reprises et informations de provenance sont agrégés dans une table de 18 colonnes.

Les deux fonctions locales extraient les valeurs distinctes non manquantes d’un groupe et les réunissent sous forme de texte.

In [25]:
def valeurs_uniques(groupe, colonne):
    return sorted(set(groupe[colonne].dropna()) - {''})


def joindre_valeurs_uniques(groupe, colonne):
    return ' | '.join(valeurs_uniques(groupe, colonne))

In [26]:
ecarts = entrees_prepare.loc[entrees_prepare.motif_ecart.ne('')].copy()
valides = entrees_prepare.loc[entrees_prepare.motif_ecart.eq('')].copy()
lignes_consolidees = []

groupes = valides.groupby('imdb_id', sort=False)
total_groupes = groupes.ngroups
debut = perf_counter()
print(f"[3.5.4] Consolidation de {total_groupes:,} films IMDb…", flush=True)
print(f"  {len(valides):,} exploitations valides | {len(ecarts):,} lignes déjà écartées", flush=True)

for numero, (imdb, groupe) in enumerate(groupes, start=1):
    if numero % 2_000 == 0:
        print(f"  Film {numero:,}/{total_groupes:,} | {len(lignes_consolidees):,} consolidés | {perf_counter() - debut:.1f} s", flush=True)

    # Un même film ne doit pas regrouper des identifiants contradictoires.
    ids_tmdb_groupe = valeurs_uniques(groupe, 'tmdb_id')
    ids_oeuvres_jp = valeurs_uniques(groupe, 'jp_id_film')
    raisons = []

    if len(ids_tmdb_groupe) > 1:
        raisons.append('Plusieurs identifiants TMDB')
    if len(ids_oeuvres_jp) > 1:
        raisons.append('Plusieurs œuvres JP pour le même IMDb')

    if raisons:
        rejet = groupe.copy()
        rejet['motif_ecart'] = ' ; '.join(raisons)
        ecarts = pd.concat([ecarts, rejet], ignore_index=True)
        continue

    statuts = set(groupe['statut_cumul'])
    statut_cumul_invalide = bool(statuts - STATUTS_CUMUL_ACCEPTES)

    cumuls_publies = (
        groupe.loc[
            groupe['statut_cumul'].eq('total_publie_jp'),
            'entrees_france_cumulees',
        ]
        .dropna()
        .unique()
    )
    cumuls_observes = (
        groupe.loc[
            groupe['statut_cumul'].eq('exploitations_observees'),
            'entrees_france_cumulees',
        ]
        .dropna()
        .unique()
    )

    entrees_retenues = pd.NA
    type_cumul = 'inconnu'

    # Les cumuls répétés ne sont pas additionnés.
    cumuls = set(cumuls_publies) | set(cumuls_observes)

    if statut_cumul_invalide:
        type_cumul = 'a_verifier'
    elif len(cumuls) > 1:
        type_cumul = 'cumuls_contradictoires'
    elif len(cumuls) == 1:
        entrees_retenues = int(next(iter(cumuls)))
        type_cumul = 'total_publie_jp' if len(cumuls_publies) > 0 else 'exploitations_observees'

    # Une saga n’est retenue que si les exploitations s’accordent.
    sagas = valeurs_uniques(groupe, 'saga')
    nb_reprises = groupe['est_reprise'].sum()

    lignes_consolidees.append({
        'tconst': imdb,
        'tmdb_id_source_entrees': ids_tmdb_groupe[0] if ids_tmdb_groupe else pd.NA,
        'entrees_france': entrees_retenues,
        'type_cumul_entrees': type_cumul,
        'statuts_cumul_source': joindre_valeurs_uniques(groupe, 'statut_cumul'),
        'saga': sagas[0] if len(sagas) == 1 else pd.NA,
        'genre_source': joindre_valeurs_uniques(groupe, 'genre_source') or pd.NA,
        'statut_saga': (
            'renseignee' if len(sagas) == 1
            else 'conflit' if len(sagas) > 1
            else 'non_renseignee'
        ),
        'a_une_reprise_referencee': nb_reprises > 0,
        'nombre_reprises_referencees': nb_reprises,
        'nombre_exploitations_referencees': len(groupe),
        'jp_ids': joindre_valeurs_uniques(groupe, 'jp_id'),
        'statuts_jointure_entrees': joindre_valeurs_uniques(groupe, 'statut_jointure'),
        'controles_oeuvre_entrees': joindre_valeurs_uniques(groupe, 'controle_oeuvre'),
        'source_entrees': 'Entrees_France.csv / JP Box-Office',
        'urls_source_entrees': joindre_valeurs_uniques(groupe, 'url_source_cumul'),
        'urls_source_saga': joindre_valeurs_uniques(groupe, 'url_source_saga'),
        'dates_collecte_entrees': joindre_valeurs_uniques(groupe, 'date_collecte'),
    })

entrees_consolidees = pd.DataFrame(lignes_consolidees)
entrees_consolidees['entrees_france'] = entrees_consolidees['entrees_france'].astype('Int64')
verifier_ids(entrees_consolidees, 'Entrées France consolidées')

print(f"[OK] {len(entrees_consolidees):,} films consolidés | {len(ecarts):,} lignes écartées au total | {perf_counter() - debut:.1f} s", flush=True)

[3.5.4] Consolidation de 17,341 films IMDb…
  17,469 exploitations valides | 128 lignes déjà écartées
  Film 2,000/17,341 | 1,999 consolidés | 5.4 s
  Film 4,000/17,341 | 3,999 consolidés | 10.8 s
  Film 6,000/17,341 | 5,999 consolidés | 16.2 s
  Film 8,000/17,341 | 7,999 consolidés | 21.7 s
  Film 10,000/17,341 | 9,999 consolidés | 27.0 s
  Film 12,000/17,341 | 11,999 consolidés | 32.4 s
  Film 14,000/17,341 | 13,999 consolidés | 37.7 s
  Film 16,000/17,341 | 15,999 consolidés | 43.1 s
[OK] 17,340 films consolidés | 130 lignes écartées au total | 46.8 s


### 3.6 — Bilan de la préparation

Comparer les volumes retenus à la lecture et ceux obtenus après préparation. La réduction du nombre de lignes comprend la sélection de fiches, le dédoublonnage et l’agrégation ; elle ne correspond pas nécessairement à des rejets de films.

In [27]:
bilan_preparation = pd.DataFrame({
    'Source': [
        'IMDb Basics',
        'IMDb Ratings',
        'TMDB',
        'IMDb Akas',
        'Entrées France',
    ],
    'Lignes après préparation': [
        len(films),
        len(notes),
        len(tmdb_prepare),
        len(titres),
        len(entrees_consolidees),
    ],
    'Nombre de colonnes': [
        films.shape[1],
        notes.shape[1],
        tmdb_prepare.shape[1],
        titres.shape[1],
        entrees_consolidees.shape[1],
    ],
})

bilan_preparation['Lignes avant préparation'] = bilan_preparation['Source'].map(
    inventaire.set_index('Source')['Lignes retenues'],
)
bilan_preparation['Réduction de lignes'] = (
    bilan_preparation['Lignes avant préparation'] - bilan_preparation['Lignes après préparation']
)
bilan_preparation = bilan_preparation[[
    'Source', 'Lignes avant préparation', 'Lignes après préparation',
    'Réduction de lignes', 'Nombre de colonnes',
]]

display(bilan_preparation)
print(f"Entrées France : {len(ecarts):,} lignes écartées après contrôle des exploitations et des groupes.")
print("[OK] Préparation des cinq sources terminée.")

,Source,Lignes avant préparation,Lignes après préparation,Réduction de lignes,Nombre de colonnes
0,IMDb Basics,761872,761872,0,8
1,IMDb Ratings,352335,352335,0,3
2,TMDB,366494,363832,2662,15
3,IMDb Akas,106162,95500,10662,2
4,Entrées France,17597,17340,257,18


Entrées France : 130 lignes écartées après contrôle des exploitations et des groupes.
[OK] Préparation des cinq sources terminée.


## 4 — Jointures et enrichissement du catalogue

Le catalogue prend IMDb Basics comme table de référence. Les jointures gauches conservent tous les films ; la cardinalité `1:1` interdit leur multiplication. Les données absentes restent manquantes. Les identifiants TMDB sont conservés sous forme de chaînes, conformément au format des sources contrôlées.

### 4.1 — Jointure IMDb Ratings

Associer les notes et le nombre de votes IMDb à chaque film.

In [28]:
print("[4.1 — IMDb Ratings]", flush=True)
catalogue = pd.merge(
    left=films,
    right=notes,
    how='left',
    left_on='tconst',
    right_on='tconst',
    validate='1:1',
)
print(f"[OK] {len(catalogue):,} films | {catalogue.shape[1]} colonnes", flush=True)

[4.1 — IMDb Ratings]
[OK] 761,872 films | 10 colonnes


### 4.2 — Jointure TMDB

Associer la fiche TMDB sélectionnée en 3.3 grâce à l’identifiant IMDb. Les fiches sans rattachement direct restent disponibles pour les compléments locaux.

In [29]:
print("[4.2 — TMDB]", flush=True)
catalogue = pd.merge(
    left=catalogue,
    right=tmdb_prepare,
    how='left',
    left_on='tconst',
    right_on='imdb_id',
    validate='1:1',
)
print(f"[OK] {len(catalogue):,} films | {catalogue.shape[1]} colonnes", flush=True)

[4.2 — TMDB]
[OK] 761,872 films | 25 colonnes


### 4.3 — Jointure IMDb Akas

Associer le titre français sélectionné en 3.4 à chaque identifiant IMDb.

In [30]:
print("[4.3 — IMDb Akas]", flush=True)
catalogue = pd.merge(
    left=catalogue,
    right=titres,
    how='left',
    left_on='tconst',
    right_on='titleId',
    validate='1:1',
)
print(f"[OK] {len(catalogue):,} films | {catalogue.shape[1]} colonnes", flush=True)

[4.3 — IMDb Akas]
[OK] 761,872 films | 27 colonnes


### 4.4 — Jointure et contrôle d’Entrées France

#### 4.4.1 — Rattachement des exploitations

Rattacher les exploitations consolidées par identifiant IMDb.

In [31]:
print("[4.4.1 — Entrées France]", flush=True)
catalogue = pd.merge(
    left=catalogue,
    right=entrees_consolidees,
    how='left',
    left_on='tconst',
    right_on='tconst',
    validate='1:1',
)
print(f"[OK] {len(catalogue):,} films | {catalogue.shape[1]} colonnes", flush=True)

[4.4.1 — Entrées France]
[OK] 761,872 films | 44 colonnes


#### 4.4.2 — Contrôle de concordance et neutralisation

En cas de contradiction entre identifiants TMDB, conserver le film mais neutraliser les données métier issues d’Entrées France. Les informations de provenance et le statut du conflit sont conservés pour le suivi. Les compteurs de reprises et d’exploitations utilisent des types acceptant les valeurs manquantes.

In [32]:
print("[4.4.2 — Contrôle des correspondances]", flush=True)
masque_conflits = (
    catalogue['tmdb_id'].notna()
    & catalogue['tmdb_id_source_entrees'].notna()
    & catalogue['tmdb_id'].ne(catalogue['tmdb_id_source_entrees'])
)
masque_correspondance = (catalogue['tconst'].isin(entrees_consolidees['tconst']))

catalogue['statut_enrichissement_entrees'] = 'sans_correspondance_exploitable'
catalogue.loc[masque_correspondance, 'statut_enrichissement_entrees'] = 'rattache_par_imdb'

champs_entrees_a_neutraliser = [
    'entrees_france',
    'saga',
    'genre_source',
    'a_une_reprise_referencee',
    'nombre_reprises_referencees',
    'nombre_exploitations_referencees',
]

# Conserver le film, mais pas les données métier d’une correspondance contradictoire.
catalogue.loc[masque_conflits, champs_entrees_a_neutraliser] = pd.NA
catalogue.loc[masque_conflits, 'statut_enrichissement_entrees'] = 'conflit_imdb_tmdb'
catalogue.loc[masque_conflits, 'type_cumul_entrees'] = 'conflit_identifiants'
catalogue.loc[masque_conflits, 'statut_saga'] = 'conflit_identifiants'

catalogue['type_cumul_entrees'] = catalogue['type_cumul_entrees'].fillna('inconnu')
catalogue['statut_saga'] = catalogue['statut_saga'].fillna('non_renseignee')

catalogue['nombre_reprises_referencees'] = catalogue['nombre_reprises_referencees'].astype('Int64')
catalogue['nombre_exploitations_referencees'] = catalogue['nombre_exploitations_referencees'].astype('Int64')
catalogue['a_une_reprise_referencee'] = catalogue['a_une_reprise_referencee'].astype('boolean')

display(catalogue['statut_enrichissement_entrees'].value_counts())

[4.4.2 — Contrôle des correspondances]


statut_enrichissement_entrees
sans_correspondance_exploitable    744658
rattache_par_imdb                   17195
conflit_imdb_tmdb                      19
Name: count, dtype: int64

### 4.5 — Compléments TMDB locaux

Récupérer les identifiants et champs manquants à partir des fiches TMDB déjà chargées, sans appel API ni remplacement des données existantes.

#### 4.5.1 — Paires vérifiées et identifiants cibles

Retenir uniquement les paires confirmées par un identifiant externe et présentes dans les données consolidées. Écarter les identifiants TMDB associés à plusieurs IMDb dans Entrées France. Donner la priorité à l’identifiant TMDB déjà présent dans le catalogue.

In [33]:
print("[4.5.1 — Paires vérifiées]", flush=True)
masque = entrees['controle_concordance'] == 'paire_confirmee_par_identifiant_externe'
paires_confirmees = entrees.loc[masque, ['imdb_id', 'tmdb_id']]

paires_confirmees = paires_confirmees.dropna()
paires_confirmees = paires_confirmees.drop_duplicates()

paires_verifiees = pd.merge(
    left=entrees_consolidees[['tconst', 'tmdb_id_source_entrees']],
    right=paires_confirmees,
    how='inner',
    left_on=['tconst', 'tmdb_id_source_entrees'],
    right_on=['imdb_id', 'tmdb_id'],
    validate='1:1',
)

liens_entrees = entrees[['imdb_id', 'tmdb_id']]
liens_entrees = liens_entrees.dropna()
liens_entrees = liens_entrees.drop_duplicates()

ids_tmdb_ambigus = liens_entrees.loc[liens_entrees['tmdb_id'].duplicated(keep=False), 'tmdb_id']
paires_verifiees = paires_verifiees.loc[~paires_verifiees['tmdb_id'].isin(ids_tmdb_ambigus)]

print(f"Identifiants TMDB ambigus : {ids_tmdb_ambigus.nunique()}")
print(f"Paires exploitables : {len(paires_verifiees)}")

id_tmdb_candidat = catalogue['tconst'].map(paires_verifiees.set_index('tconst')['tmdb_id'])
id_tmdb_cible = catalogue['tmdb_id'].combine_first(id_tmdb_candidat)
print(f"Identifiants TMDB manquants avec candidat : {(catalogue['tmdb_id'].isna() & id_tmdb_cible.notna()).sum():,}")

[4.5.1 — Paires vérifiées]
Identifiants TMDB ambigus : 0
Paires exploitables : 17313
Identifiants TMDB manquants avec candidat : 629


#### 4.5.2 — Contrôle des identifiants cibles

Identifier les collisions dans le CSV TMDB et retrouver l’IMDb propriétaire de chaque identifiant cible. Ces informations servent à déterminer les compléments autorisés.

In [34]:
print("[4.5.2 — Contrôle des identifiants]", flush=True)
liens_tmdb = tmdb[['id', 'imdb_id']]
liens_tmdb = liens_tmdb.dropna()
liens_tmdb = liens_tmdb.drop_duplicates()

collisions_tmdb = liens_tmdb.loc[liens_tmdb['id'].duplicated(keep=False), 'id']

print(f"Identifiants TMDB associés à plusieurs IMDb : {collisions_tmdb.nunique()}")

imdb_par_id_tmdb = (
    liens_tmdb
    .drop_duplicates(subset='id')
    .set_index('id')['imdb_id']
)
id_imdb_connu = id_tmdb_cible.map(imdb_par_id_tmdb)

[4.5.2 — Contrôle des identifiants]
Identifiants TMDB associés à plusieurs IMDb : 0


#### 4.5.3 — Autorisation et récupération des identifiants

Interdire les compléments en cas de contradiction, d’affectation d’un TMDB à plusieurs films ou de conflit lors de la jointure Entrées France. Compléter uniquement les identifiants absents et autorisés, en conservant leur provenance.

In [35]:
print("[4.5.3 — Autorisation et récupération des identifiants]", flush=True)
masque_conflit_tmdb = (
    id_tmdb_cible.isin(collisions_tmdb)
    | (id_imdb_connu.notna() & id_imdb_connu.ne(catalogue['tconst']))
)

masque_doubles_affectations = (id_tmdb_cible.duplicated(keep=False)) & (id_tmdb_cible.notna())

masque_conflit_tmdb = (
    masque_conflit_tmdb
    | (catalogue['statut_enrichissement_entrees'] == 'conflit_imdb_tmdb')
    | masque_doubles_affectations
)

masque_complement_autorise = (id_tmdb_cible.notna()) & (~masque_conflit_tmdb)

masque_ajout_id_tmdb = masque_complement_autorise & catalogue['tmdb_id'].isna()

catalogue.loc[masque_ajout_id_tmdb, 'tmdb_id'] = id_tmdb_cible[masque_ajout_id_tmdb]

# Distinguer les identifiants directs des identifiants récupérés.
catalogue['source_tmdb_id'] = pd.Series(pd.NA, index=catalogue.index, dtype='string')
catalogue.loc[catalogue['tmdb_id'].notna(), 'source_tmdb_id'] = 'CSV TMDB / IMDb'
catalogue.loc[masque_ajout_id_tmdb, 'source_tmdb_id'] = 'Entrees_France.csv / paire verifiee'

print(f"Conflits interdisant les compléments : {masque_conflit_tmdb.sum():,}")
print(f"Identifiants TMDB récupérés : {masque_ajout_id_tmdb.sum():,}")
display(catalogue['source_tmdb_id'].value_counts(dropna=False))

[4.5.3 — Autorisation et récupération des identifiants]
Conflits interdisant les compléments : 40
Identifiants TMDB récupérés : 611


source_tmdb_id
<NA>                                   397429
CSV TMDB / IMDb                        363832
Entrees_France.csv / paire verifiee       611
Name: count, dtype: int64[pyarrow]

#### 4.5.4 — Complément des champs manquants

Sélectionner une fiche par identifiant TMDB en privilégiant le nombre de votes le plus élevé (ordre stable en cas d’égalité). Compléter les champs vides uniquement si la fiche est disponible et autorisée. Conserver les statuts, la liste des champs complétés et les compteurs avant traitement. L’identifiant IMDb canonique reste celui d’IMDb Basics.

In [36]:
print("[4.5.4 — Complément des champs]", flush=True)
tmdb_complement = tmdb.copy()
tmdb_complement['vote_count'] = pd.to_numeric(tmdb_complement['vote_count'], errors='coerce')
tmdb_complement = tmdb_complement.sort_values('vote_count', ascending=False, kind='mergesort')
tmdb_complement = tmdb_complement.dropna(subset=['id'])
tmdb_complement = tmdb_complement.drop_duplicates('id')

tmdb_complement = tmdb_complement.rename(columns={
    'id': 'tmdb_id',
    'title': 'titre_tmdb',
    'vote_count': 'votes_tmdb',
    'vote_average': 'note_tmdb',
})

tmdb_complement = tmdb_complement.set_index('tmdb_id')
champs_a_completer = tmdb_complement.columns.drop('imdb_id')

masque_fiche_initiale = catalogue['tmdb_id'].notna() & ~masque_ajout_id_tmdb
masque_fiche_complement = masque_complement_autorise & id_tmdb_cible.isin(tmdb_complement.index)

# Les statuts distinguent les fiches directes, récupérées et non exploitables.
catalogue['fiche_tmdb_disponible'] = masque_fiche_initiale | masque_fiche_complement
catalogue['champs_completes_tmdb'] = ""
catalogue['statut_complement_tmdb'] = 'sans_paire_verifiee'
catalogue.loc[masque_fiche_initiale, 'statut_complement_tmdb'] = 'fiche_csv_par_imdb'
catalogue.loc[masque_conflit_tmdb, 'statut_complement_tmdb'] = 'conflit_identifiants_aucun_complement'
catalogue.loc[masque_ajout_id_tmdb, 'statut_complement_tmdb'] = 'identifiant_seul_csv_absent'
catalogue.loc[masque_ajout_id_tmdb & masque_fiche_complement, 'statut_complement_tmdb'] = 'fiche_csv_par_paire_verifiee'
catalogue.loc[masque_ajout_id_tmdb, 'champs_completes_tmdb'] = 'tmdb_id'

# L’identifiant a déjà été complété en 4.5.3 : reconstituer son compteur initial.
vides_avant = {}
vides_avant['tmdb_id'] = (
    catalogue['tmdb_id'].isna().sum()
    + masque_ajout_id_tmdb.sum()
)

# Compléter exclusivement les champs vides avec une valeur disponible.
for champ in champs_a_completer:
    masque_vide = champ_vide(catalogue[champ])
    vides_avant[champ] = masque_vide.sum()
    valeurs_complement = id_tmdb_cible.map(tmdb_complement[champ])
    masque = (masque_fiche_complement) & masque_vide & (~champ_vide(valeurs_complement))
    catalogue.loc[masque, champ] = valeurs_complement
    catalogue.loc[masque, 'champs_completes_tmdb'] += ' | ' + champ
    print(f"  {champ} : {masque.sum():,} valeurs complétées", flush=True)


catalogue['champs_completes_tmdb'] = catalogue['champs_completes_tmdb'].str.strip(' |')
catalogue['imdb_id'] = catalogue['tconst']
print('[OK] Compléments TMDB terminés.', flush=True)

[4.5.4 — Complément des champs]
  titre_tmdb : 611 valeurs complétées
  note_tmdb : 611 valeurs complétées
  votes_tmdb : 611 valeurs complétées
  release_date : 87 valeurs complétées
  revenue : 611 valeurs complétées
  budget : 611 valeurs complétées
  original_language : 611 valeurs complétées
  original_title : 611 valeurs complétées
  overview : 240 valeurs complétées
  popularity : 611 valeurs complétées
  poster_path : 95 valeurs complétées
  production_countries : 448 valeurs complétées
  keywords : 88 valeurs complétées
[OK] Compléments TMDB terminés.


#### 4.5.5 — Bilan des compléments TMDB

Comparer le nombre de valeurs manquantes avant et après les compléments, puis afficher la répartition des statuts. Aucun champ du catalogue n’est modifié ici.

In [37]:
vides_apres = {}

for champ in vides_avant:
    vides_apres[champ] = champ_vide(catalogue[champ]).sum()

bilan_complement_tmdb = pd.DataFrame([
    {
        'Champ': champ,
        'Vides avant': vides_avant[champ],
        'Completes': vides_avant[champ] - vides_apres[champ],
        'Vides apres': vides_apres[champ],
    }
    for champ in vides_avant
])

display(bilan_complement_tmdb)

print("\nStatuts des compléments TMDB :")
display(catalogue['statut_complement_tmdb'].value_counts())

,Champ,Vides avant,Completes,Vides apres
0,tmdb_id,398040,611,397429
1,titre_tmdb,398042,611,397431
2,note_tmdb,398040,611,397429
3,votes_tmdb,398040,611,397429
4,release_date,434428,87,434341
5,revenue,398040,611,397429
6,budget,398040,611,397429
7,original_language,398040,611,397429
8,original_title,398041,611,397430
9,overview,447806,240,447566



Statuts des compléments TMDB :


statut_complement_tmdb
sans_paire_verifiee                      397411
fiche_csv_par_imdb                       363810
fiche_csv_par_paire_verifiee                611
conflit_identifiants_aucun_complement        40
Name: count, dtype: int64

### 4.6 — Classification des productions

Distinguer les productions exclusivement françaises, les coproductions, les films sans France renseignée et les origines inconnues. Les masques vectorisés reproduisent la classification initiale sur les valeurs distinctes des sources contrôlées ; ils supposent des libellés de pays normalisés comme dans ces fichiers. La classification détaillée sert à construire `origine_production` et `film_francais`, les deux colonnes conservées à l’export.

In [38]:
print("[4.6 — Origine des productions]", flush=True)
masque_inconnu = champ_vide(catalogue['production_countries'])
masque_france = catalogue['production_countries'] == 'France'
masque_coproduction = (
    ~masque_france
    & catalogue['production_countries'].str.contains('France', na=False, regex=False)
)

catalogue['type_production_francaise'] = 'Sans France renseignée'
catalogue.loc[masque_inconnu, 'type_production_francaise'] = 'Origine inconnue'
catalogue.loc[masque_france, 'type_production_francaise'] = 'Production française'
catalogue.loc[masque_coproduction, 'type_production_francaise'] = 'Coproduction française'

catalogue['origine_production'] = catalogue['type_production_francaise'].map({
    'Production française': 'Prod_100%_France',
    'Coproduction française': 'Co_production_Française',
    'Sans France renseignée': 'Étrangère',
    'Origine inconnue': 'Origine inconnue',
})
catalogue['film_francais'] = catalogue['origine_production'].isin(
    ['Prod_100%_France', 'Co_production_Française'],
)

display(catalogue['type_production_francaise'].value_counts(dropna=False))

[4.6 — Origine des productions]


type_production_francaise
Origine inconnue          498884
Sans France renseignée    244670
Production française        9725
Coproduction française      8593
Name: count, dtype: int64

### 4.7 — Bilan des jointures

Mesurer la couverture de chaque enrichissement et vérifier la conservation de tous les identifiants IMDb. Pour Entrées France, seules les correspondances validées sont comptées ; les conflits restent dans le catalogue sans enrichissement métier. Les autres indicateurs mesurent la présence des données utiles, pas nécessairement toutes les correspondances de clés.

In [39]:
bilan_jointures = pd.DataFrame([
    {
        'Source': 'IMDb Ratings',
        'Correspondances': catalogue['averageRating'].notna().sum(),
    },
    {
        'Source': 'TMDB - Jointure initiale',
        'Correspondances': masque_fiche_initiale.sum(),
    },
    {
        'Source': 'TMDB - Après compléments',
        'Correspondances': catalogue['fiche_tmdb_disponible'].sum(),
    },
    {
        'Source': 'IMDb Akas - Titres FR',
        'Correspondances': catalogue['titre_fr'].notna().sum(),
    },
    {
        'Source': 'Entrées France',
        'Correspondances': catalogue['statut_enrichissement_entrees'].eq('rattache_par_imdb').sum(),
    },
])

bilan_jointures['Sans correspondance'] = len(catalogue) - bilan_jointures['Correspondances']
bilan_jointures['Couverture (%)'] = (
    bilan_jointures['Correspondances'] / len(catalogue) * 100
).round(2)

display(bilan_jointures)

print(f"Nombre de films : {len(catalogue):,}")
print(f"Nombre d'identifiants IMDb uniques : {catalogue['tconst'].nunique():,}")

assert len(catalogue) == len(films)
assert catalogue['tconst'].is_unique
assert catalogue['tconst'].notna().all()
assert catalogue['tconst'].tolist() == films['tconst'].tolist()
print("[OK] Jointures et enrichissement terminés : aucun film perdu, dupliqué ou réordonné.")

,Source,Correspondances,Sans correspondance,Couverture (%)
0,IMDb Ratings,352335,409537,46.25
1,TMDB - Jointure initiale,363832,398040,47.76
2,TMDB - Après compléments,364443,397429,47.84
3,IMDb Akas - Titres FR,95500,666372,12.53
4,Entrées France,17195,744677,2.26


Nombre de films : 761,872
Nombre d'identifiants IMDb uniques : 761,872
[OK] Jointures et enrichissement terminés : aucun film perdu, dupliqué ou réordonné.


## 5 — Construction du titre d’affichage

Retenir le titre français lorsqu’il est disponible, puis le titre principal IMDb, le titre original et, en dernier recours, l’identifiant IMDb.

In [40]:
catalogue['titre_affiche'] = catalogue['titre_fr'].combine_first(
    catalogue['primaryTitle'],
)
catalogue['titre_affiche'] = catalogue['titre_affiche'].combine_first(
    catalogue['originalTitle'],
)
catalogue['titre_affiche'] = catalogue['titre_affiche'].fillna(catalogue['tconst'])

## 6 — Application des filtres métier

Appliquer les critères de `FILTRES_METIER` au catalogue enrichi. Le catalogue complet reste disponible ; chaque filtre indique le nombre de films conservés et écartés. Une borne à `None` désactive la limite correspondante et une liste de critères vide désactive le filtre associé. Les votes minimaux des films français et non français sont indépendants.

### 6.1 — Fonctions de filtrage

Les fonctions construisent des masques sur les intervalles et genres, puis appliquent chaque masque en affichant le bilan de sélection.

In [41]:
def filtre_intervalle(serie, minimum=None, maximum=None):
    masque = pd.Series(True, index=serie.index)
    if minimum is not None:
        masque &= serie >= minimum
    if maximum is not None:
        masque &= serie <= maximum

    return masque


def filtre_genres(serie, genres):
    masque = pd.Series(False, index=serie.index)

    if genres is None:
        return masque

    for genre in genres:
        masque |= serie.str.contains(genre, na=False, regex=False, case=False)

    return masque


def appliquer_filtre(df, masque, nom):
    avant = df.shape[0]
    df = df[masque]
    apres = df.shape[0]
    print(f"{nom} : {avant} -> {apres} films ({avant - apres} écartés)")

    return df

### 6.2 — Sélection des films

Les critères sont appliqués successivement. Les entrées inconnues sont traitées selon la configuration ; les genres sont recherchés sans distinction de casse.

In [42]:
catalogue_filtre = catalogue.copy()
print(f"Catalogue initial : {len(catalogue_filtre)} films\n")

# Année de sortie
if FILTRES_METIER['annee_min'] is not None or FILTRES_METIER['annee_max'] is not None:
    masque = filtre_intervalle(
        catalogue_filtre['startYear'],
        FILTRES_METIER['annee_min'],
        FILTRES_METIER['annee_max'],
    )
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Année de sortie")
else:
    print("Année de sortie : filtre désactivé")

# Durée
if FILTRES_METIER['duree_min'] is not None or FILTRES_METIER['duree_max'] is not None:
    masque = filtre_intervalle(
        catalogue_filtre['runtimeMinutes'],
        FILTRES_METIER['duree_min'],
        FILTRES_METIER['duree_max'],
    )
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Durée")
else:
    print("Durée : filtre désactivé")

# Entrées France
masque = filtre_intervalle(
    catalogue_filtre['entrees_france'],
    FILTRES_METIER['entrees_min'],
    FILTRES_METIER['entrees_max'],
)

if FILTRES_METIER['garder_entrees_inconnues']:
    masque |= catalogue_filtre['entrees_france'].isna()
else:
    masque &= catalogue_filtre['entrees_france'].notna()

catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Entrées France")

# Genres exclus
if FILTRES_METIER['genres_exclus']:
    masque = ~filtre_genres(
        catalogue_filtre['genre_source'],
        FILTRES_METIER['genres_exclus'],
    )
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Genres exclus")
else:
    print("Genres exclus : filtre désactivé")

# Genres inclus
if FILTRES_METIER['genres_inclus']:
    masque = filtre_genres(
        catalogue_filtre['genre_source'],
        FILTRES_METIER['genres_inclus'],
    )
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Genres inclus")
else:
    print("Genres inclus : filtre désactivé")

# Contenus adultes
if FILTRES_METIER['non_adultes']:
    masque = catalogue_filtre['isAdult'] == 0
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Films non adultes")
else:
    print("Films non adultes : filtre désactivé")

# Langues exclues
if FILTRES_METIER['langues_exclues']:
    masque = ~catalogue_filtre['original_language'].isin(
        FILTRES_METIER['langues_exclues'],
    )
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Langues")
else:
    print("Langues : filtre désactivé")

# Origines de production
if FILTRES_METIER['origines']:
    masque = catalogue_filtre['origine_production'].isin(
        FILTRES_METIER['origines'],
    )
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Origines")
else:
    print("Origines : filtre désactivé")

# Votes IMDb - Films non français
if FILTRES_METIER['votes_min'] is not None:
    masque = (
        catalogue_filtre['film_francais']
        | (catalogue_filtre['numVotes'] >= FILTRES_METIER['votes_min'])
    )
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Votes IMDb - Non français")
else:
    print("Votes IMDb - Non français : filtre désactivé")

# Votes IMDb - Films français
if FILTRES_METIER['votes_min_fr'] is not None:
    masque = (
        ~catalogue_filtre['film_francais']
        | (catalogue_filtre['numVotes'] >= FILTRES_METIER['votes_min_fr'])
    )
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Votes IMDb - Français")
else:
    print("Votes IMDb - Français : filtre désactivé")

# Disponibilité TMDB
if FILTRES_METIER['exiger_tmdb']:
    masque = catalogue_filtre['fiche_tmdb_disponible']
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Disponibilité TMDB")
else:
    print("Disponibilité TMDB : filtre désactivé")

# Affiche et résumé
if FILTRES_METIER['exiger_visuels']:
    masque = (
        catalogue_filtre['poster_path'].fillna('').str.strip().ne('')
        & catalogue_filtre['overview'].fillna('').str.strip().ne('')
    )
    catalogue_filtre = appliquer_filtre(catalogue_filtre, masque, "Visuels")
else:
    print("Visuels : filtre désactivé")

print(f"\nCatalogue final : {len(catalogue_filtre)} films")

Catalogue initial : 761872 films

Année de sortie : 761872 -> 354498 films (407374 écartés)
Durée : 354498 -> 243854 films (110644 écartés)
Entrées France : 243854 -> 7412 films (236442 écartés)
Genres exclus : 7412 -> 7357 films (55 écartés)
Genres inclus : 7357 -> 7357 films (0 écartés)
Films non adultes : 7357 -> 7356 films (1 écartés)
Langues : filtre désactivé
Origines : filtre désactivé
Votes IMDb - Non français : filtre désactivé
Votes IMDb - Français : filtre désactivé
Disponibilité TMDB : filtre désactivé
Visuels : filtre désactivé

Catalogue final : 7356 films


## 7 — Validation du catalogue filtré

Vérifier que la sélection contient au moins un film. Les identifiants IMDb ont déjà été contrôlés en section 4.7 ; le filtrage ne fait que sélectionner des lignes du catalogue.

In [43]:
assert not catalogue_filtre.empty, "Aucun film ne correspond aux critères métier."

## 8 — Export du catalogue

Exporter les 18 colonnes définies dans `COLONNES_EXPORT`, dans l’ordre configuré et sans index pandas. Le fichier `data/processed/catalogue.csv` est le seul résultat exporté par ce notebook. L’enrichissement en acteurs et réalisateurs relève d’un traitement distinct.

In [44]:
catalogue_filtre[COLONNES_EXPORT].to_csv(CHEMIN_CATALOGUE, index=False)
print(f'[OK] Catalogue exporté : {CHEMIN_CATALOGUE}')
print(f'{len(catalogue_filtre):,} films | {len(COLONNES_EXPORT)} colonnes')

[OK] Catalogue exporté : f:\Antoine\Documents\forward-to-the-past\data\processed\catalogue.csv
7,356 films | 18 colonnes
